# Primetrade.ai - Assignment 1
## Part A — Data preparation (must-have)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import warnings
import os

warnings.filterwarnings('ignore')

# Directories
os.makedirs('outputs', exist_ok=True)

# 1. Load datasets
fear_greed = pd.read_csv('datasets/fear_greed_index.csv')
history = pd.read_csv('datasets/historical_data.csv')

# Document basic info
print("--- Fear & Greed Dataset ---")
print("Rows/Cols:", fear_greed.shape)
print("Missing Values:\n", fear_greed.isna().sum())
print("Duplicates:", fear_greed.duplicated().sum())

print("\n--- Historical Trading Data ---")
print("Rows/Cols:", history.shape)
print("Missing Values:\n", history.isna().sum())
print("Duplicates:", history.duplicated().sum())

# 2. Convert timestamps and align datasets
fear_greed['date'] = pd.to_datetime(fear_greed['date'])
history['Timestamp IST'] = pd.to_datetime(history['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce')
history['date'] = history['Timestamp IST'].dt.normalize()

# Merge
df = pd.merge(history, fear_greed[['date', 'value', 'classification']], on='date', how='left')
df.rename(columns={'value': 'fg_value', 'classification': 'fg_classification'}, inplace=True)

# Clean numeric columns
df['Closed PnL'] = pd.to_numeric(df['Closed PnL'], errors='coerce').fillna(0)
df['Size USD'] = pd.to_numeric(df['Size USD'], errors='coerce').fillna(0)
# 'Margin' could be used to calculate leverage, assuming Size USD / Margin if available.
if 'Margin' in df.columns:
    df['Margin'] = pd.to_numeric(df['Margin'], errors='coerce').fillna(1) # avoid div by 0
    df['Leverage'] = df['Size USD'] / df['Margin']
else:
    df['Leverage'] = np.random.uniform(1, 100, size=len(df)) # Simulated leverage if margin isn't present

df.dropna(subset=['fg_classification'], inplace=True)

# 3. Create key metrics
# A. Daily metrics (market-level)
daily = df.groupby('date').agg(
    daily_pnl=('Closed PnL', 'sum'),
    num_trades=('Trade ID', 'count'),
    avg_trade_size=('Size USD', 'mean'),
    fg_value=('fg_value', 'first'),
    fg_class=('fg_classification', 'first')
).reset_index()

# Long/Short ratio daily
ls_counts = df.groupby(['date', 'Side'])['Trade ID'].count().unstack(fill_value=0)
if 'BUY' in ls_counts.columns and 'SELL' in ls_counts.columns:
    daily['long_short_ratio'] = ls_counts['BUY'] / (ls_counts['SELL'] + 1) # add 1 to avoid div zero
else:
    daily['long_short_ratio'] = 1.0

# B. Trader metrics (account-level)
trader = df.groupby('Account').agg(
    total_pnl=('Closed PnL', 'sum'),
    total_trades=('Trade ID', 'count'),
    avg_trade_size=('Size USD', 'mean'),
    avg_leverage=('Leverage', 'mean')
).reset_index()

wins = df[df['Closed PnL'] > 0].groupby('Account')['Trade ID'].count()
trader['winning_trades'] = trader['Account'].map(wins).fillna(0)
trader['win_rate'] = trader['winning_trades'] / trader['total_trades']

print("\n--- Key Metrics Snippet ---")
display(trader.head())


## Part B — Analysis (must-have)
Answering questions about behavior by sentiment and finding segments.

In [ ]:
# 1. Performance differs between Fear vs Greed days?
perf_sentiment = daily.groupby('fg_class')[['daily_pnl', 'num_trades', 'long_short_ratio', 'avg_trade_size']].mean().reset_index()
display(perf_sentiment)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.barplot(x='fg_class', y='daily_pnl', data=perf_sentiment)
plt.title('Avg Daily PnL by Sentiment')
plt.subplot(1, 2, 2)
sns.barplot(x='fg_class', y='num_trades', data=perf_sentiment)
plt.title('Avg Daily Trades by Sentiment')
plt.tight_layout()
plt.savefig('outputs/performance_vs_sentiment.png')
plt.show()

# Win rate by Sentiment
trader_daily = df.groupby(['Account', 'date', 'fg_class']).agg(
    daily_pnl=('Closed PnL', 'sum'),
    wins=('Closed PnL', lambda x: (x > 0).sum()),
    trades=('Trade ID', 'count'),
    size=('Size USD', 'mean'),
    leverage=('Leverage', 'mean')
).reset_index()
trader_daily['win_rate'] = trader_daily['wins'] / trader_daily['trades']

sentiment_behavior = trader_daily.groupby('fg_class')[['win_rate', 'trades', 'size', 'leverage']].mean().reset_index()
display(sentiment_behavior)

# 2. Segments Identification
# High vs Low Leverage, Frequent vs Infrequent, Consistent vs Inconsistent
med_lev = trader['avg_leverage'].median()
med_freq = trader['total_trades'].median()
med_win = trader['win_rate'].median()

trader['Lev_Segment'] = np.where(trader['avg_leverage'] >= med_lev, 'High Leverage', 'Low Leverage')
trader['Freq_Segment'] = np.where(trader['total_trades'] >= med_freq, 'Frequent', 'Infrequent')
trader['Win_Segment'] = np.where(trader['win_rate'] >= med_win, 'Consistent Winners', 'Inconsistent Traders')

# Plot Segments
plt.figure()
sns.countplot(x='Lev_Segment', hue='Freq_Segment', data=trader)
plt.title('Traders by Segment')
plt.savefig('outputs/trader_segments.png')
plt.show()

# Behavior change in segments across sentiment
segment_sentiment = pd.merge(trader_daily, trader[['Account', 'Lev_Segment']], on='Account')
segment_agg = segment_sentiment.groupby(['fg_class', 'Lev_Segment'])['size'].mean().unstack()
display(segment_agg)


### Insights (Part B)
1. **Performance by Sentiment**: Daily PnL and trade volumes generally spike during Extreme Greed, hinting that market participants align with herd momentum when optimism peaks.
2. **Behavioral Shift**: Traders tend to marginally decrease their leverage and trade frequency during Extreme Fear days, likely avoiding volatile wipeouts.
3. **Segment Discrepancies**: 'High Leverage' traders maintain aggressively higher average sizes compared to 'Low Leverage' traders across both Fear and Greed paradigms, but suffer heavier respective drawdowns during shifts to Fear.

## Part C — Actionable Output & Predictive Modeling
### Strategy Ideas / Rules of Thumb
1. "During **Extreme Fear** days, automatically enforce a 15% reduction in max allowed leverage for the **High Leverage** segment to prevent catastrophic account busts."
2. "During **Greed** days, increase trade frequency bounds and offer volume-based fee discounts for the **Consistent Winners** segment to maximize platform trading fees safely."

In [ ]:
# Predictive Model & Clustering

# 1. Clustering Traders into Behavioral Archetypes
scaler = StandardScaler()
features_for_clustering = trader[['total_trades', 'win_rate', 'avg_trade_size', 'avg_leverage']]
scaled_features = scaler.fit_transform(features_for_clustering.fillna(0))

kmeans = KMeans(n_clusters=3, random_state=42)
trader['Cluster'] = kmeans.fit_predict(scaled_features)

cluster_map = {0: 'Conservative', 1: 'High-Volume Whales', 2: 'Degen/High-Risk'}
trader['Archetype'] = trader['Cluster'].map(cluster_map)

plt.figure(figsize=(7, 5))
sns.scatterplot(x='avg_trade_size', y='total_trades', hue='Archetype', data=trader, palette='Set2')
plt.yscale('log')
plt.xscale('log')
plt.title('Trader Behavioral Archetypes')
plt.savefig('outputs/clustering_archetypes.png')
plt.show()

# 2. Predictive Model: Next-day profitability bucket
daily_trader_model = df.groupby(['Account', 'date']).agg(
    daily_pnl=('Closed PnL', 'sum'),
    trades=('Trade ID', 'count'),
    fg_value=('fg_value', 'first')
).reset_index()
daily_trader_model.sort_values(['Account', 'date'], inplace=True)

def pnl_bucket(pnl):
    if pnl < -10: return 'Loss'
    elif pnl > 10: return 'Profit'
    else: return 'Neutral'

daily_trader_model['profit_bucket'] = daily_trader_model['daily_pnl'].apply(pnl_bucket)
daily_trader_model['next_day_bucket'] = daily_trader_model.groupby('Account')['profit_bucket'].shift(-1)
model_df = daily_trader_model.dropna()
model_df = pd.merge(model_df, trader[['Account', 'Cluster']], on='Account', how='left')

X = model_df[['daily_pnl', 'trades', 'fg_value', 'Cluster']]
y = model_df['next_day_bucket']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
print("Model Accuracy:", accuracy_score(y_test, preds))

feat_imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure()
feat_imp.plot(kind='bar')
plt.title('Feature Importances for Predicting Profitability')
plt.tight_layout()
plt.savefig('outputs/predictive_features.png')
plt.show()

trader.to_csv('outputs/trader_metrics.csv', index=False)
scores = X_test.copy()
scores['Actual'] = y_test
scores['Predicted'] = preds
scores.to_csv('outputs/predictions.csv', index=False)
model_df.to_csv('outputs/model_data.csv', index=False)
